# LLM-based Toxicity Moderation: lightweight models

Few-shot conversation-level toxicity classification with lightweight LLMs, evaluated per-instance (Local LLM setup) and on a shared global test set.

**Models:** `dolphin3:8b` (local, via Ollama), Claude Haiku 4.5, GPT-5.4 mini, Gemini 3.1 Flash-Lite.
Switch between them with `LLM_PROVIDER` in Section 2.

**Datasets:** Pleroma and Mastodon. Set the paths in Section 4.

**Protocol:** for each seed in `{1, 42, 999}`, 10 instances are sampled. Each builds its own 20-example few-shot prompt (10 toxic + 10 non-toxic) and is evaluated on (a) its own 100-conversation local test set and (b) a shared 100-conversation global test set scored by all 10 instances. Results are reported as mean ± std across seeds.

**Conversation filter:** only conversations with at least `MIN_POSTS_PER_CONVERSATION` posts are used, matching the conversation-length filter applied in the federated GNN experiments.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Install dependencies

In [ ]:
!pip -q install requests pandas numpy scikit-learn tqdm

## 2. LLM provider switch

Set `LLM_PROVIDER` to one of `"anthropic"`, `"openai"`, `"gemini"`, `"ollama"`, then set the model and API key for whichever you picked.

API keys: use the **Secrets** panel in Colab and add `ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `GEMINI_API_KEY` there and toggle notebook access on. Or just paste a key directly into the `api_key` field below, or set it as an environment variable.

In [ ]:
import os

# get API keys from Colab
try:
    from google.colab import userdata
    def _secret(name):
        try:
            return userdata.get(name) or ""
        except Exception:
            return os.environ.get(name, "")
except Exception:
    def _secret(name):
        return os.environ.get(name, "")

LLM_PROVIDER = "ollama" # "anthropic" / "openai" / "gemini" / "ollama"

PROVIDERS = {
    "anthropic": {
        "model": "claude-haiku-4-5",
        "api_key": _secret("ANTHROPIC_API_KEY"),
        "temperature": 0,
        "max_tokens": 128,
        "cache_prompt": True,
    },
    "openai": {
        "model": "gpt-5.4-mini",
        "api_key": _secret("OPENAI_API_KEY"),
        "reasoning_effort": "none",
        "temperature": None,
        "max_tokens": 2048,
    },
    "gemini": {
        "model": "gemini-3.1-flash-lite",
        "api_key": _secret("GEMINI_API_KEY"),
        "thinking_level": "low",
        "temperature": None,
        "max_tokens": 2048,
    },
    "ollama": {
        "model": "dolphin3:8b",
        "url": "http://localhost:11434/api/generate",
        "temperature": 0,
        "max_tokens": 128,
    },
}

print("Active provider:", LLM_PROVIDER)
print("Model:", PROVIDERS[LLM_PROVIDER]["model"])

## 2b. Cost / speed / rate-limit settings

These knobs control the optimizations. The defaults keep the paper setup: no input truncation and one request at a time.

- `OUTPUT_MODE`: `"label_only"` is cheaper/faster. `"label_and_reason"` is the prompt used in the paper.
- `MAX_WORKERS`: parallel requests. Leave at 1 if you hit 429s, raise only if your API tier allows more requests/min.
- `REQUESTS_PER_MINUTE`: client-side throttle to stay under your tier limit (the main 429 prevention).
- `MAX_CONV_CHARS` / `MAX_FEWSHOT_EXAMPLE_CHARS`: optional input caps (`None` = off). Capping changes the text the model sees, so leave off unless you need the extra savings.

In [ ]:
OUTPUT_MODE = "label_and_reason" # "label_only" / "label_and_reason"

# input caps (capping changes model input)
MAX_CONV_CHARS = None # truncate each conversation being classified
MAX_FEWSHOT_EXAMPLE_CHARS = None  # truncate each few-shot example

# concurrency & rate limiting (HTTP 429 mitigation)
MAX_WORKERS = 1 # parallel requests
REQUESTS_PER_MINUTE = 50 # client-side cap
MAX_RETRIES = 8
BACKOFF_BASE = 1.5 # base seconds for exponential backoff
BACKOFF_CAP = 60.0 # max seconds to wait on single backoff

print(f"OUTPUT_MODE={OUTPUT_MODE} | workers={MAX_WORKERS} | rpm={REQUESTS_PER_MINUTE} | retries={MAX_RETRIES}")

## 2c. Token-usage capture

Shared schema written to `usage_log.csv` during the run, consumed by the cost cell at the end. This cell is identical in both notebooks.

In [ ]:
import pandas as pd
from pathlib import Path

USAGE_KEYS = ["provider", "model", "dataset", "seed", "variant", "eval_type"]
USAGE_SUMS = ["n_requests", "input_tokens", "cached_read_tokens", "cache_write_tokens", "output_tokens", "reasoning_tokens"]


def normalize_usage(provider, usage):
    u = usage or {}
    if provider == "anthropic":
        read = u.get("cache_read_input_tokens", 0) or 0
        write = u.get("cache_creation_input_tokens", 0) or 0
        return dict(input_tokens=(u.get("input_tokens", 0) or 0) + read + write,
                    cached_read_tokens=read, cache_write_tokens=write, output_tokens=u.get("output_tokens", 0) or 0, reasoning_tokens=0)
    if provider == "openai":
        return dict(input_tokens=u.get("prompt_tokens", 0) or 0,
                    cached_read_tokens=(u.get("prompt_tokens_details") or {}).get("cached_tokens", 0) or 0,
                    cache_write_tokens=0, output_tokens=u.get("completion_tokens", 0) or 0,
                    reasoning_tokens=(u.get("completion_tokens_details") or {}).get("reasoning_tokens", 0) or 0)
    if provider == "gemini":
        thoughts = u.get("thoughtsTokenCount", 0) or 0
        return dict(input_tokens=u.get("promptTokenCount", 0) or 0,
                    cached_read_tokens=u.get("cachedContentTokenCount", 0) or 0,
                    cache_write_tokens=0, output_tokens=(u.get("candidatesTokenCount", 0) or 0) + thoughts, reasoning_tokens=thoughts)
    if provider == "ollama":
        return dict(input_tokens=u.get("prompt_eval_count", 0) or 0,
                    cached_read_tokens=0, cache_write_tokens=0, output_tokens=u.get("eval_count", 0) or 0, reasoning_tokens=0)
    return dict(input_tokens=0, cached_read_tokens=0, cache_write_tokens=0, output_tokens=0, reasoning_tokens=0)


def flush_usage_log(rows, path, batch_api):
    """Aggregate per-request usage rows and merge them into usage_log.csv."""
    path = Path(path)
    if not rows:
        print("no usage rows to write")
        return None
    new = pd.DataFrame(rows).groupby(USAGE_KEYS, as_index=False)[USAGE_SUMS].sum()
    new["batch_api"] = bool(batch_api)

    if path.exists():
        merged = pd.concat([pd.read_csv(path), new], ignore_index=True)
        merged = merged.drop_duplicates(subset=USAGE_KEYS, keep="last")
    else:
        merged = new
    path.parent.mkdir(parents=True, exist_ok=True)
    merged.sort_values(USAGE_KEYS).to_csv(path, index=False)
    print(f"usage log: {len(new)} job rows written, {len(merged)} total -> {path}")
    return merged

print("usage capture ready")

## 3. Unified `call_llm` dispatcher

Each provider gets a small function that takes a prompt string and returns the raw text response. `call_llm(prompt)` routes to the active provider. The rest of the notebook only ever calls `call_llm`, so nothing else needs to change when you switch backends.

In [ ]:
import os, time, shutil, subprocess, requests

# 0. the installer needs zstd to extract its archive
!apt-get -qq install -y zstd

# 1. install
!curl -fsSL https://ollama.com/install.sh -o /tmp/ollama_install.sh
!sh /tmp/ollama_install.sh

# 2. locate the binary
OLLAMA = shutil.which("ollama")
if OLLAMA is None:
    for p in ["/usr/local/bin/ollama", "/usr/bin/ollama", "/root/.ollama/bin/ollama"]:
        if os.path.exists(p):
            OLLAMA = p
            break
if OLLAMA is None:
    raise RuntimeError("ollama binary still not found - read the install output above")
print("using:", OLLAMA)

# 3. start the server in the background
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
log = open("/tmp/ollama.log", "w")
subprocess.Popen([OLLAMA, "serve"], stdout=log, stderr=log)

# 4. wait until it answers
for i in range(90):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("ollama is up after", i, "s")
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/tmp/ollama.log").read()[-2000:])
    raise RuntimeError("server did not start - see log above")

# 5. pull the model (several GB, a few minutes)
subprocess.run([OLLAMA, "pull", "dolphin3:8b"], check=True)

print(requests.get("http://127.0.0.1:11434/api/tags").json())

In [ ]:
import time
import random
import requests
import threading

# client-side rate limit (min interval between requests)
class _RateLimiter:
    def __init__(self, rpm):
        self.min_interval = 60.0 / rpm if rpm and rpm > 0 else 0.0
        self._lock = threading.Lock()
        self._next = 0.0

    def wait(self):
        if self.min_interval <= 0:
            return
        with self._lock:
            now = time.monotonic()
            sleep = max(0.0, self._next - now)
            self._next = max(now, self._next) + self.min_interval
        if sleep > 0:
            time.sleep(sleep)

_rate_limiter = _RateLimiter(REQUESTS_PER_MINUTE)

# per-request token accounting
PROMPT_VARIANT = "base"
_USAGE_ROWS = []
_USAGE_CTX = {"dataset": None, "seed": None, "eval_type": None}
_usage_lock = threading.Lock()

def set_usage_context(dataset, seed, eval_type):
    with _usage_lock:
        _USAGE_CTX.update(dataset=dataset, seed=seed, eval_type=eval_type)

def _record_usage(provider, model, usage):
    row = normalize_usage(provider, usage)
    with _usage_lock:
        _USAGE_ROWS.append({
            "provider": provider, "model": model,
            "dataset": _USAGE_CTX["dataset"], "seed": _USAGE_CTX["seed"],
            "variant": PROMPT_VARIANT, "eval_type": _USAGE_CTX["eval_type"],
            "n_requests": 1, **row,
        })

def call_anthropic(system_text, user_text, cfg):
    system_block = [{"type": "text", "text": system_text}]
    if cfg.get("cache_prompt", True):
        system_block[0]["cache_control"] = {"type": "ephemeral"}
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={
            "x-api-key": cfg["api_key"],
            "anthropic-version": "2023-06-01",
            "content-type": "application/json",
        },
        json={
            "model": cfg["model"],
            "max_tokens": cfg.get("max_tokens", 128),
            "temperature": cfg.get("temperature", 0),
            "system": system_block,
            "messages": [{"role": "user", "content": user_text}],
        },
        timeout=180,
    )
    resp.raise_for_status()
    data = resp.json()
    _record_usage("anthropic", cfg["model"], data.get("usage"))
    return "".join(b.get("text", "") for b in data.get("content", []) if b.get("type") == "text").strip()

def call_openai(system_text, user_text, cfg):
    body = {
        "model": cfg["model"],
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_text},
        ],
        "max_completion_tokens": cfg.get("max_tokens", 2048),
    }
    if cfg.get("reasoning_effort"):
        body["reasoning_effort"] = cfg["reasoning_effort"]
    if cfg.get("temperature") is not None:
        body["temperature"] = cfg["temperature"]

    resp = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={"Authorization": f"Bearer {cfg['api_key']}", "Content-Type": "application/json"},
        json=body,
        timeout=180,
    )
    resp.raise_for_status()
    data = resp.json()
    _record_usage("openai", cfg["model"], data.get("usage"))
    return data["choices"][0]["message"]["content"].strip()

def call_gemini(system_text, user_text, cfg):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{cfg['model']}:generateContent"

    gen_cfg = {
        "maxOutputTokens": cfg.get("max_tokens", 2048),
        "responseMimeType": "text/x.enum",
        "responseSchema": {"type": "STRING", "enum": ["TOXIC", "NON-TOXIC"]},
    }
    if cfg.get("temperature") is not None:
        gen_cfg["temperature"] = cfg["temperature"]
    if cfg.get("thinking_level"):
        gen_cfg["thinkingConfig"] = {"thinkingLevel": cfg["thinking_level"]}

    safety = [
        {"category": c, "threshold": "BLOCK_NONE"}
        for c in (
            "HARM_CATEGORY_HARASSMENT",
            "HARM_CATEGORY_HATE_SPEECH",
            "HARM_CATEGORY_SEXUALLY_EXPLICIT",
            "HARM_CATEGORY_DANGEROUS_CONTENT",
            "HARM_CATEGORY_CIVIC_INTEGRITY",
        )
    ]

    resp = requests.post(
        url,
        headers={"x-goog-api-key": cfg["api_key"], "Content-Type": "application/json"},
        json={
            "systemInstruction": {"parts": [{"text": system_text}]},
            "contents": [{"parts": [{"text": user_text}]}],
            "generationConfig": gen_cfg,
            "safetySettings": safety,
        },
        timeout=180,
    )
    resp.raise_for_status()
    data = resp.json()
    _record_usage("gemini", cfg["model"], data.get("usageMetadata"))

    # whole prompt blocked -> no candidates
    candidates = data.get("candidates")
    if not candidates:
        reason = (data.get("promptFeedback") or {}).get("blockReason", "no candidates returned")
        raise RuntimeError(f"Gemini returned no candidates (blockReason={reason})")

    cand = candidates[0]
    finish = cand.get("finishReason")
    parts = (cand.get("content") or {}).get("parts", []) or []
    text = "".join(p.get("text", "") for p in parts).strip()
    if not text:
        raise RuntimeError(f"Gemini returned empty text (finishReason={finish})")
    return text

def call_ollama(system_text, user_text, cfg):
    resp = requests.post(
        cfg["url"],
        json={
            "model": cfg["model"],
            "system": system_text,
            "prompt": user_text,
            "stream": False,
            "options": {"temperature": cfg.get("temperature", 0)},
        },
        timeout=180,
    )
    resp.raise_for_status()
    data = resp.json()
    _record_usage("ollama", cfg["model"], data)
    return data.get("response", "").strip()

_DISPATCH = {
    "anthropic": call_anthropic,
    "openai": call_openai,
    "gemini": call_gemini,
    "ollama": call_ollama,
}

def _is_retryable(exc):
    if isinstance(exc, requests.HTTPError) and exc.response is not None:
        return exc.response.status_code in (408, 409, 429, 500, 502, 503, 504, 529)
    return isinstance(exc, (requests.ConnectionError, requests.Timeout))

def _retry_after_seconds(exc):
    try:
        ra = exc.response.headers.get("retry-after")
        return float(ra) if ra is not None else None
    except Exception:
        return None

def call_llm(system_text, user_text):
    """Send (system-user) to active provider with rate limiting + backoff on 429/transient errors"""
    cfg = PROVIDERS[LLM_PROVIDER]
    if LLM_PROVIDER != "ollama" and not cfg.get("api_key"):
        raise RuntimeError(f"No API key set for provider '{LLM_PROVIDER}'")
    fn = _DISPATCH[LLM_PROVIDER]

    last_exc = None
    for attempt in range(MAX_RETRIES + 1):
        _rate_limiter.wait()
        try:
            return fn(system_text, user_text, cfg)
        except Exception as exc:
            last_exc = exc
            if attempt >= MAX_RETRIES or not _is_retryable(exc):
                raise
            wait = _retry_after_seconds(exc)
            if wait is None:
                wait = min(BACKOFF_CAP, BACKOFF_BASE * (2 ** attempt)) * (0.5 + random.random())
            time.sleep(wait)
    raise last_exc

# sanity check that the LLM responds
print("Test:", call_llm("You are a test assistant.", "Reply with the single word: OK"))

## 4. Experiment configuration

The local test set uses 50 Toxic + 50 Non-Toxic conversations per client. The whole pipeline runs once per seed in `RANDOM_STATES`.

In [ ]:
from pathlib import Path

DATASET_NAME = "pleroma" # "pleroma" / "mastodon"

DATA_CSV = "<PATH_TO>/{ds}_data/{ds}_data_after_filtering.csv".format(ds=DATASET_NAME)
LABELS_CSV = "<PATH_TO>/{ds}_data/conversation_labels.csv".format(ds=DATASET_NAME)
OUTPUT_BASE = Path("<PATH_TO>/{ds}_data/local_llm_results".format(ds=DATASET_NAME))
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

RANDOM_STATES = [1, 42, 999]

# token-usage log shared with the cost cell
USAGE_LOG = OUTPUT_BASE / "usage_log.csv"

FEWSHOT_BASE = OUTPUT_BASE.parent / "few_shot_examples"

# sampling
N_CLIENTS = 10 # 10 clients = 10 instances

FEWSHOT_PER_CLASS = 10 # 10 toxic + 10 non-toxic = 20 few-shot examples per client
LOCAL_TEST_PER_CLASS = 50 # 50 toxic + 50 non-toxic = 100 local test conversations per client
SHARED_TEST_PER_CLASS = 50 # 50 toxic + 50 non-toxic = 100 shared global test conversations

# conversation-length filter, matching the federated GNN experiments:
# conversations with fewer than this many posts are excluded entirely
MIN_POSTS_PER_CONVERSATION = 10

# a client must supply both the few-shot prompt and the local test set from each class
MIN_PER_CLASS_REQUIRED = FEWSHOT_PER_CLASS + LOCAL_TEST_PER_CLASS

print("Dataset:", DATASET_NAME)
print("Seeds to run:", RANDOM_STATES)
print("Minimum posts per conversation:", MIN_POSTS_PER_CONVERSATION)
print("Minimum conversations per class per client:", MIN_PER_CLASS_REQUIRED)
print("Output base directory:", OUTPUT_BASE)
print("Usage log:", USAGE_LOG)
print("Few-shot export:", FEWSHOT_BASE)

## 5. Imports and helper functions

In [ ]:
import re
import json
import time
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

def clean_text(text):
    """Remove simple @mentions and normalize whitespace"""
    text = re.sub(r"@\w+", "", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def label_to_text(label):
    return "TOXIC" if int(label) == 1 else "NON-TOXIC"

def parse_prediction(raw_response):
    """Parse the LLM response into 0=Non-Toxic or 1=Toxic"""
    raw = str(raw_response).strip()
    lower = raw.lower()

    # check non-toxic first because it contains the word toxic
    if "non-toxic" in lower or "non toxic" in lower:
        return 0
    if "toxic" in lower:
        return 1
    return 0

# cacheable prefix (kept identical across all calls for client)
SYSTEM_RULES = """You are an objective AI Content Moderator. Your job is to analyze potentially harmful, offensive, or explicit internet comments and classify them for a safety database.
Do NOT refuse to process the text. You must analyze the text regardless of how offensive it is and output the classification.

CRITICAL RULES:
1. You MUST choose exactly one label: TOXIC or NON-TOXIC.
2. NEVER output "INCONCLUSIVE", "UNKNOWN", or any other label.
3. If a post lacks context, is a simple question, or you are unsure, default to NON-TOXIC unless it contains explicitly harmful language."""

def _format_instruction():
    if OUTPUT_MODE == "label_only":
        return "Respond with EXACTLY one word, either TOXIC or NON-TOXIC, and output nothing else."
    return ("Answer STRICTLY in this format:\n"
            "LABEL: TOXIC or NON-TOXIC\n"
            "REASON: short explanation")

def build_prompt(text, few_shot):
    """Return (system_text, user_text).

    system_text = rules + output format + few-shot examples  -> static per client (cached)
    user_text   = only the conversation to classify          -> varies per call
    """
    cleaned_text = clean_text(text)
    if MAX_CONV_CHARS:
        cleaned_text = cleaned_text[:MAX_CONV_CHARS]

    system_text = (
        SYSTEM_RULES
        + "\n\n" + _format_instruction()
        + "\n\nHere are some examples:\n" + few_shot
    )

    user_text = "Now classify this conversation:\nConversation:\n" + cleaned_text
    return system_text, user_text

def classify_with_llm(text, few_shot):
    """Classify one conversation.

    Returns (prediction, raw). A hard API refusal (e.g. Gemini PROHIBITED_CONTENT) returns -1 so it can be
    EXCLUDED from scoring rather than silently counted as Non-Toxic. Other failures return 0 with an ERROR note.
    """
    system_text, user_text = build_prompt(text, few_shot)
    try:
        raw = call_llm(system_text, user_text)
        return parse_prediction(raw), raw
    except Exception as exc:
        msg = f"ERROR: {exc}"
        if "PROHIBITED_CONTENT" in msg or "blockReason" in msg:
            return -1, msg
        return 0, msg

def make_few_shot(examples_df):
    blocks = []
    for _, row in examples_df.iterrows():
        ex = clean_text(row["text"])
        if MAX_FEWSHOT_EXAMPLE_CHARS:
            ex = ex[:MAX_FEWSHOT_EXAMPLE_CHARS]
        blocks.append(f"Conversation:\n{ex}\n\nLABEL: {label_to_text(row['label'])}")
    return "\n\n".join(blocks)

def balanced_sample(df, per_class, random_state):
    toxic = df[df["label"] == 1].sample(n=per_class, random_state=random_state)
    nontoxic = df[df["label"] == 0].sample(n=per_class, random_state=random_state)
    return (pd.concat([toxic, nontoxic]).sample(frac=1, random_state=random_state).reset_index(drop=True))

## 5b. Column check

Prints the columns found in the posts and labels files and auto-detects the instance column. Confirm the printed choice before continuing. Set `INSTANCE_COL` by hand if the guess is wrong.

In [ ]:
import pandas as pd

_probe_posts = pd.read_csv(DATA_CSV, nrows=5)
_probe_labels = pd.read_csv(LABELS_CSV, nrows=5)

print("POSTS columns :", list(_probe_posts.columns))
print("LABELS columns:", list(_probe_labels.columns))
print()

# instance column -> auto-detect
INSTANCE_COL = None  # "instance_toot_found_id"

if INSTANCE_COL is None:
    preferred = ["instance_toot_found_id", "instance_toot_id", "instance_id", "instance"]
    for cand in preferred:
        if cand in _probe_posts.columns:
            INSTANCE_COL = cand
            break
    if INSTANCE_COL is None:
        matches = [c for c in _probe_posts.columns if "instance" in c.lower()]
        if len(matches) == 1:
            INSTANCE_COL = matches[0]
        elif matches:
            raise ValueError(
                f"Several instance-like columns found: {matches}. "
                f"Set INSTANCE_COL manually above."
            )
        else:
            raise ValueError("No instance column found. Set INSTANCE_COL manually above.")

TEXT_COL = "text"
CONV_COL = "conversation_id"

print(">>> Using INSTANCE_COL =", INSTANCE_COL)
print(">>> Using CONV_COL =", CONV_COL)
print(">>> Using TEXT_COL =", TEXT_COL)

# quick look at how many instances / conversations this gives
_full = pd.read_csv(DATA_CSV, usecols=[CONV_COL, INSTANCE_COL])
print()
print("Unique conversations in posts file:", _full[CONV_COL].nunique())
print("Unique instances in posts file:", _full[INSTANCE_COL].nunique())
del _full

## 6. Load and build conversation-level dataset

This is seed-independent, so it runs once (~3mins).

In [ ]:
df_posts = pd.read_csv(DATA_CSV)
df_labels = pd.read_csv(LABELS_CSV)

required_post_cols = {CONV_COL, INSTANCE_COL, TEXT_COL}
required_label_cols = {"conversation_id", "label"}

missing_posts = required_post_cols - set(df_posts.columns)
missing_labels = required_label_cols - set(df_labels.columns)

if missing_posts:
    raise ValueError(f"Missing columns in posts file: {missing_posts}")
if missing_labels:
    raise ValueError(f"Missing columns in labels file: {missing_labels}")

labels_clean = df_labels[["conversation_id", "label"]].drop_duplicates("conversation_id").copy()
valid_conversations = set(labels_clean["conversation_id"])

posts_valid = df_posts[df_posts[CONV_COL].isin(valid_conversations)].copy()

# conversation-length filter (only conversations with at least MIN_POSTS_PER_CONVERSATION posts are kept)
posts_per_conv = posts_valid.groupby(CONV_COL).size()
long_enough = posts_per_conv[posts_per_conv >= MIN_POSTS_PER_CONVERSATION].index

n_before = posts_valid[CONV_COL].nunique()
posts_valid = posts_valid[posts_valid[CONV_COL].isin(long_enough)].copy()
n_after = posts_valid[CONV_COL].nunique()

print(f"Conversation-length filter (>= {MIN_POSTS_PER_CONVERSATION} posts): "
      f"{n_before:,} -> {n_after:,} conversations "
      f"({n_before - n_after:,} removed)")

conv_text = (posts_valid.groupby([CONV_COL, INSTANCE_COL], as_index=False)[TEXT_COL].agg(lambda values: "\n---\n".join(values.astype(str))))

conv_labels = conv_text.merge(labels_clean, on="conversation_id", how="inner")
conv_labels["label"] = conv_labels["label"].astype(int)

# normalise the instance column name so the rest of the notebook is dataset-agnostic
if INSTANCE_COL != "instance_toot_found_id":
    conv_labels = conv_labels.rename(columns={INSTANCE_COL: "instance_toot_found_id"})

print("Conversation rows:", len(conv_labels))
print("Instances:", conv_labels["instance_toot_found_id"].nunique())
print("Label distribution:")
print(conv_labels["label"].value_counts().rename(index={0: "Non-Toxic", 1: "Toxic"}))

## 7. Filter usable clients

The candidate pool (clients with enough Toxic and Non-Toxic conversations) is seed-independent, so it is computed once. Which `N_CLIENTS` get sampled from this pool is seed-dependent and happens inside `run_experiment`.

In [ ]:
def summarize_instances(conv_df):
    rows = []
    for instance_id, df_i in conv_df.groupby("instance_toot_found_id"):
        rows.append({
            "instance_id": instance_id,
            "total": len(df_i),
            "toxic": int((df_i["label"] == 1).sum()),
            "nontoxic": int((df_i["label"] == 0).sum()),
        })
    return pd.DataFrame(rows)

instance_summary = summarize_instances(conv_labels)

candidate_df = instance_summary[(instance_summary["toxic"] >= MIN_PER_CLASS_REQUIRED) & (instance_summary["nontoxic"] >= MIN_PER_CLASS_REQUIRED)].copy()

print("Candidate clients after variety filtering:", len(candidate_df))
print("Required per class:", MIN_PER_CLASS_REQUIRED)
display(candidate_df.sort_values(["toxic", "nontoxic"], ascending=False).head(20))

if len(candidate_df) < N_CLIENTS:
    raise ValueError(
        f"Only {len(candidate_df)} clients satisfy the requirement of at least "
        f"{MIN_PER_CLASS_REQUIRED} Toxic and {MIN_PER_CLASS_REQUIRED} Non-Toxic conversations. "
        f"Cannot select {N_CLIENTS} clients. Lower LOCAL_TEST_PER_CLASS or use more data."
    )

## 8. Evaluation helpers

In [ ]:
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

def evaluate_rows(rows_df, few_shot, model_instance_id, eval_type, random_state):
    set_usage_context(DATASET_NAME, random_state, eval_type)
    rows = rows_df.reset_index(drop=True)
    total = len(rows)
    results = [None] * total

    def work(item):
        i, row = item
        pred, raw = classify_with_llm(row["text"], few_shot)
        return i, pred, raw

    items = list(rows.iterrows())
    desc = f"rs{random_state} | {eval_type} | {model_instance_id}"

    if MAX_WORKERS and MAX_WORKERS > 1:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            for i, pred, raw in tqdm(ex.map(work, items), total=total, desc=desc, leave=False):
                results[i] = (pred, raw)
    else:
        for item in tqdm(items, total=total, desc=desc, leave=False):
            i, pred, raw = work(item)
            results[i] = (pred, raw)

    records = []
    for i, row in items:
        pred, raw = results[i]
        records.append({
            "random_state": random_state,
            "eval_type": eval_type,
            "model_instance_id": model_instance_id,
            "test_instance_id": row["instance_toot_found_id"],
            "conversation_id": row["conversation_id"],
            "ground_truth": int(row["label"]),
            "prediction": int(pred),
            "raw_response": raw,
            "text": row["text"],
        })
    return pd.DataFrame(records)

def summarize_predictions(pred_df, name):
    # exclude API refusals from metrics (prediction == -1)
    scored = pred_df[pred_df["prediction"] != -1]
    n_excluded = len(pred_df) - len(scored)

    y_true = scored["ground_truth"].astype(int)
    y_pred = scored["prediction"].astype(int)

    return {
        "name": name,
        "n": len(scored),
        "n_excluded": n_excluded,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "toxic_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "toxic_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

def count_errors(pred_df, tag):
    raw = pred_df["raw_response"].astype(str)
    n_err = raw.str.startswith("ERROR").sum()
    n_refused = int((pred_df["prediction"] == -1).sum())
    if n_err:
        print(f"WARNING: {tag}: {n_err}/{len(pred_df)} calls failed. "
              f"{n_refused} were API refusals (excluded from scoring). "
              f"{n_err - n_refused} were other errors (scored as Non-Toxic). Inspect 'raw_response'.")
    return int(n_err)

def print_report(pred_df, title):
    scored = pred_df[pred_df["prediction"] != -1]
    n_excluded = len(pred_df) - len(scored)
    print("=" * 80)
    print(title)
    if n_excluded:
        print(f"(excluded {n_excluded} API-refused rows from metrics)")
    print("=" * 80)
    print(classification_report(
        scored["ground_truth"].astype(int),
        scored["prediction"].astype(int),
        labels=[0, 1],
        target_names=["Non-Toxic", "Toxic"],
        zero_division=0
    ))

    cm = confusion_matrix(
        scored["ground_truth"].astype(int),
        scored["prediction"].astype(int),
        labels=[0, 1]
    )
    print("Confusion matrix [rows=true, cols=pred]:")
    print(pd.DataFrame(cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]))
    return cm

## 9. Per-seed pipeline

`run_experiment(random_state)` does everything that depends on the seed: pick clients, build balanced few-shot prompts and local test sets, build the shared global test set, run both evaluations, write reports, and save all CSVs to `OUTPUT_BASE / f"rs{random_state}"`. It returns the summaries so we can aggregate across seeds afterwards.

In [ ]:
def run_experiment(random_state):
    out_dir = OUTPUT_BASE / f"rs{random_state}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"# RUNNING SEED rs{random_state} -> {out_dir}")

    # select clients (seed dependent)
    selected_clients = (
        candidate_df["instance_id"]
        .sample(n=N_CLIENTS, random_state=random_state)
        .tolist()
    )
    print("Selected clients:", len(selected_clients))
    print(selected_clients)

    # build balanced prompts + local test sets for each client
    client_data = {}
    setup_rows = []

    for instance_id in selected_clients:
        df_i = conv_labels[conv_labels["instance_toot_found_id"] == instance_id].copy()

        fewshot_df = balanced_sample(df_i, FEWSHOT_PER_CLASS, random_state)
        used_ids = set(fewshot_df["conversation_id"])

        remaining_df = df_i[~df_i["conversation_id"].isin(used_ids)].copy()
        local_test_df = balanced_sample(remaining_df, LOCAL_TEST_PER_CLASS, random_state)

        few_shot = make_few_shot(fewshot_df)

        client_data[instance_id] = {
            "fewshot_df": fewshot_df,
            "few_shot": few_shot,
            "local_test_df": local_test_df,
        }

        setup_rows.append({
            "instance_id": instance_id,
            "fewshot_total": len(fewshot_df),
            "fewshot_toxic": int((fewshot_df["label"] == 1).sum()),
            "fewshot_nontoxic": int((fewshot_df["label"] == 0).sum()),
            "local_test_total": len(local_test_df),
            "local_test_toxic": int((local_test_df["label"] == 1).sum()),
            "local_test_nontoxic": int((local_test_df["label"] == 0).sum()),
        })

    setup_df = pd.DataFrame(setup_rows)
    print("Setup summary totals:")
    print(setup_df.drop(columns=["instance_id"]).sum())
    setup_df.to_csv(out_dir / "client_setup_summary.csv", index=False)

    # persist the few-shot prompts: the SOTA batch runner reuses these verbatim
    fewshot_dir = FEWSHOT_BASE / f"rs{random_state}"
    fewshot_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([{"random_state": random_state, "model_instance_id": instance_id, "few_shot": client_data[instance_id]["few_shot"]}
                  for instance_id in selected_clients]).to_csv(fewshot_dir / "few_shot_examples.csv", index=False)
    
    print("Saved few-shot prompts to:", fewshot_dir / "few_shot_examples.csv")

    # build balanced shared global test set
    all_fewshot_ids = set()
    for data in client_data.values():
        all_fewshot_ids.update(data["fewshot_df"]["conversation_id"].tolist())

    shared_pool = conv_labels[conv_labels["instance_toot_found_id"].isin(selected_clients)].copy()
    shared_pool = shared_pool[~shared_pool["conversation_id"].isin(all_fewshot_ids)].copy()
    shared_global_test_df = balanced_sample(shared_pool, SHARED_TEST_PER_CLASS, random_state)

    print("Shared global test size:", len(shared_global_test_df))
    shared_global_test_df.to_csv(out_dir / "shared_global_test_set.csv", index=False)

    # local evaluation (each LLM on its own local test set)
    local_prediction_frames = []
    local_per_client_rows = []

    for instance_id in selected_clients:
        few_shot = client_data[instance_id]["few_shot"]
        local_test_df = client_data[instance_id]["local_test_df"]

        pred_df = evaluate_rows(
            rows_df=local_test_df,
            few_shot=few_shot,
            model_instance_id=instance_id,
            eval_type="local_test",
            random_state=random_state,
        )

        local_prediction_frames.append(pred_df)
        row = summarize_predictions(pred_df, name=f"local_{instance_id}")
        row["instance_id"] = instance_id
        local_per_client_rows.append(row)

        # incremental save in case the runtime disconnects
        pd.concat(local_prediction_frames, ignore_index=True).to_csv(out_dir / "local_predictions_partial.csv", index=False)
        pd.DataFrame(local_per_client_rows).to_csv(out_dir / "local_per_client_summary_partial.csv", index=False)

    local_predictions_df = pd.concat(local_prediction_frames, ignore_index=True)
    local_per_client_summary_df = pd.DataFrame(local_per_client_rows)
    local_predictions_df.to_csv(out_dir / "local_predictions.csv", index=False)
    local_per_client_summary_df.to_csv(out_dir / "local_per_client_summary.csv", index=False)
    count_errors(local_predictions_df, f"rs{random_state} local")
    print("Local evaluation complete.")

    # shared global evaluation (each LLM on the same shared test set)
    shared_prediction_frames = []
    shared_per_client_rows = []

    for instance_id in selected_clients:
        few_shot = client_data[instance_id]["few_shot"]

        pred_df = evaluate_rows(
            rows_df=shared_global_test_df,
            few_shot=few_shot,
            model_instance_id=instance_id,
            eval_type="shared_global_test",
            random_state=random_state,
        )

        shared_prediction_frames.append(pred_df)
        row = summarize_predictions(pred_df, name=f"shared_global_{instance_id}")
        row["instance_id"] = instance_id
        shared_per_client_rows.append(row)

        # incremental save
        pd.concat(shared_prediction_frames, ignore_index=True).to_csv(out_dir / "shared_global_predictions_partial.csv", index=False)
        pd.DataFrame(shared_per_client_rows).to_csv(out_dir / "shared_global_per_client_summary_partial.csv", index=False)

    shared_global_predictions_df = pd.concat(shared_prediction_frames, ignore_index=True)
    shared_global_per_client_summary_df = pd.DataFrame(shared_per_client_rows)
    shared_global_predictions_df.to_csv(out_dir / "shared_global_predictions.csv", index=False)
    shared_global_per_client_summary_df.to_csv(out_dir / "shared_global_per_client_summary.csv", index=False)
    count_errors(shared_global_predictions_df, f"rs{random_state} shared")
    print("Shared global evaluation complete.")

    # reports + confusion matrices
    local_cm = print_report(local_predictions_df, f"[rs{random_state}] Local evaluation: each LLM on its own local test set")
    shared_cm = print_report(shared_global_predictions_df, f"[rs{random_state}] Shared global evaluation: each LLM on the same shared test set")

    final_summary_df = pd.DataFrame([
        summarize_predictions(local_predictions_df, "Local LLMs on local test sets"),
        summarize_predictions(shared_global_predictions_df, "Local LLMs on shared global test set"),
    ])

    final_summary_df.loc[0, "macro_avg_accuracy_across_clients"] = local_per_client_summary_df["accuracy"].mean()
    final_summary_df.loc[0, "macro_avg_f1_across_clients"] = local_per_client_summary_df["macro_f1"].mean()
    final_summary_df.loc[1, "macro_avg_accuracy_across_clients"] = shared_global_per_client_summary_df["accuracy"].mean()
    final_summary_df.loc[1, "macro_avg_f1_across_clients"] = shared_global_per_client_summary_df["macro_f1"].mean()
    final_summary_df.insert(0, "random_state", random_state)

    display(final_summary_df)
    final_summary_df.to_csv(out_dir / "final_summary.csv", index=False)

    pd.DataFrame(local_cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]).to_csv(out_dir / "local_confusion_matrix.csv")
    pd.DataFrame(shared_cm, index=["True Non-Toxic", "True Toxic"], columns=["Pred Non-Toxic", "Pred Toxic"]).to_csv(out_dir / "shared_global_confusion_matrix.csv")

    flush_usage_log(_USAGE_ROWS, USAGE_LOG, batch_api=False)
    _USAGE_ROWS.clear()

    print("Saved outputs to:", out_dir)

    return {
        "random_state": random_state,
        "output_dir": out_dir,
        "final_summary_df": final_summary_df,
        "local_per_client_summary_df": local_per_client_summary_df,
        "shared_global_per_client_summary_df": shared_global_per_client_summary_df,
        "local_cm": local_cm,
        "shared_cm": shared_cm,
    }

## 10. Run all seeds

This runs the full pipeline once per seed. With the default settings each seed makes roughly `N_CLIENTS x (LOCAL_TEST_PER_CLASS + SHARED_TEST_PER_CLASS) x 2 = 2000` API calls, so all three seeds are ~6000 calls.

In [ ]:
all_results = {}

for rs in RANDOM_STATES:
    all_results[rs] = run_experiment(rs)

print("\nAll seeds complete:", list(all_results.keys()))

## 11. Aggregate across seeds

Stacks each seed's final summary into one table and reports the mean and standard deviation across seeds for both evaluation types.

In [ ]:
aggregate_df = pd.concat([res["final_summary_df"] for res in all_results.values()], ignore_index=True)

display(aggregate_df)
aggregate_df.to_csv(OUTPUT_BASE / "aggregate_summary_all_seeds.csv", index=False)

# mean / std across seeds, per evaluation type
metric_cols = ["accuracy", "macro_f1", "micro_f1", "weighted_f1", "toxic_precision", "toxic_recall", "toxic_f1", "macro_avg_accuracy_across_clients", "macro_avg_f1_across_clients"]
mean_std_df = (aggregate_df.groupby("name")[metric_cols].agg(["mean", "std"]).round(4))

print("Mean/std across seeds:")
display(mean_std_df)
mean_std_df.to_csv(OUTPUT_BASE / "aggregate_mean_std_across_seeds.csv")
print("Saved aggregate summaries to:", OUTPUT_BASE)

## 12. Cost per prompt & dataset

Prices `usage_log.csv` per dataset, model and prompt variant. Reads only the log, so it makes no API calls and can be re-run at any time.

In [ ]:
import pandas as pd
from pathlib import Path

USAGE_COLUMNS = USAGE_KEYS + USAGE_SUMS + ["batch_api"]

def load_usage_log(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No usage log at {path}. Re-run the experiment cells, or point USAGE_LOG at the log you have.")
    df = pd.read_csv(path)
    missing = [c for c in USAGE_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"usage log {path} is missing columns: {missing}")
    for c in USAGE_SUMS + ["batch_api"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

def usage_per_prompt_and_dataset(usage_log_path, by=("dataset", "model", "variant")):
    df = load_usage_log(usage_log_path)
    agg = (df.groupby(list(by), as_index=False)
             .agg(n_requests=("n_requests", "sum"),
                  input_tokens=("input_tokens", "sum"),
                  cached_read_tokens=("cached_read_tokens", "sum"),
                  cache_write_tokens=("cache_write_tokens", "sum"),
                  output_tokens=("output_tokens", "sum"),
                  reasoning_tokens=("reasoning_tokens", "sum"),
                  batch_api=("batch_api", "max")))

    agg["uncached_input_tokens"] = (agg["input_tokens"] - agg["cached_read_tokens"] - agg["cache_write_tokens"]).clip(lower=0)
    agg["total_tokens"] = agg["input_tokens"] + agg["output_tokens"]
    n = agg["n_requests"].replace(0, pd.NA)
    agg["input_per_conv"] = agg["input_tokens"] / n
    agg["output_per_conv"] = agg["output_tokens"] / n
    agg["pct_input_cached"] = agg["cached_read_tokens"] / agg["input_tokens"].replace(0, pd.NA) * 100
    agg["pct_output_reasoning"] = agg["reasoning_tokens"] / agg["output_tokens"].replace(0, pd.NA) * 100
    agg["batch_api"] = agg["batch_api"].astype(bool)
    return agg.sort_values(list(by)).reset_index(drop=True)

def print_usage_report(agg):
    show = agg.copy()
    for c in ["input_tokens", "uncached_input_tokens", "cached_read_tokens",
              "output_tokens", "reasoning_tokens", "total_tokens"]:
        show[c] = show[c].map(lambda v: f"{int(v):,}")
    for c in ["input_per_conv", "output_per_conv"]:
        show[c] = show[c].map(lambda v: f"{v:,.0f}" if pd.notna(v) else "-")
    for c in ["pct_input_cached", "pct_output_reasoning"]:
        show[c] = show[c].map(lambda v: f"{v:.1f}%" if pd.notna(v) else "-")

    cols = [c for c in ["dataset", "model", "variant", "batch_api", "n_requests",
                        "input_tokens", "cached_read_tokens", "pct_input_cached",
                        "output_tokens", "reasoning_tokens", "pct_output_reasoning",
                        "total_tokens", "input_per_conv", "output_per_conv"]

            if c in show.columns]
    print(show[cols].to_string(index=False))

    tot = agg[["n_requests", "input_tokens", "cached_read_tokens", "output_tokens", "reasoning_tokens", "total_tokens"]].sum()

    print(f"\nTOTAL: {int(tot['n_requests']):,} classifications | "
          f"input {int(tot['input_tokens']):,} "
          f"(cached {int(tot['cached_read_tokens']):,}) | "
          f"output {int(tot['output_tokens']):,} "
          f"(reasoning {int(tot['reasoning_tokens']):,}) | "
          f"total {int(tot['total_tokens']):,}")

usage_table = usage_per_prompt_and_dataset(USAGE_LOG)
print_usage_report(usage_table)
usage_table.to_csv(Path(USAGE_LOG).with_name("usage_summary.csv"), index=False)